# ROC-AUC curves for every combination of the RenoRisk genes

Same LODO design, models and preprocessing as `train_ML_RR.ipynb` (cells copied unchanged except where noted).
For every non-empty subset of the 5 RenoRisk genes (C4BPA, COL13A1, HABP2, CFH, CFI) — 5 + 10 + 10 + 5 + 1 = 31
subsets — the features are the subset **plus `Compartment` and `Sex`**. All 17 PyCaret-default classifiers are run
under LODO; the best one by mean ROC-AUC (dummy excluded) is plotted as the mean of the per-cohort ROC curves.

- **Output:** `Results/Report/ROC_AUC_images_corr/ROC_AUC_<k>genes_<GENE_GENE...>_<AUC>.png` (31 images;
  `<AUC>` = the mean ROC-AUC in the image title, 3 decimals),
  `ROC_AUC_summary.xlsx` (best model and its mean metrics per subset) and `ROC_AUC_sessionInfo.txt`.
- **Caveat:** picking the best of 17 models per subset on the same LODO scores is mildly optimistic, as in the parent notebook.
- **Input:** `Data/ML_input/all_corr.xlsx` (ComBat-corrected; 1322 samples, 19 series, 17 LODO cohorts —
  NEPTUNE merged as in `train_ML_RR.ipynb`; C4BPA NA in 60 rows, mean-imputed inside each fold). The correction used every sample's disease label, so these LODO scores are optimistic compared with `all.xlsx`.

In [9]:
import sys, platform, warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import lightgbm
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
MERGE_NEPTUNE = True       # GSE108109 + GSE200828 + GSE219185 -> one LODO group (shared NEPTUNE patients)

DATA_PATH = Path('../Data/ML_input/all_corr.xlsx')  # ComBat-corrected all.xlsx: label leakage, LODO scores optimistic
OUT_DIR = Path('../Results/Report/ROC_AUC_images_corr')
OUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ['C4BPA', 'COL13A1', 'HABP2', 'CFH', 'CFI']
CATEGORICAL = ['Compartment', 'Sex']
TARGET = 'Disease_Label'

In [10]:
df = pd.read_excel(DATA_PATH)

# NEPTUNE: GSE200828 reuses 96 GSE108109 samples (Results/SampleOverlap); GSE219185 is the tubulointerstitium
# of NEPTUNE patients whose glomeruli are in GSE108109. Held out together, so no patient is on both sides of a fold.
NEPTUNE = {'GSE108109', 'GSE200828', 'GSE219185'}
df['Cohort'] = np.where(MERGE_NEPTUNE & df['StudyID'].isin(NEPTUNE), 'NEPTUNE', df['StudyID'])

X = df[GENES + CATEGORICAL]
y = df[TARGET].to_numpy()
groups = df['Cohort'].to_numpy()

print(df.shape, '| folds:', df['Cohort'].nunique())
pd.crosstab(df['Cohort'], df[TARGET], margins=True)

(850, 14) | folds: 10


Disease_Label,0,1,All
Cohort,,,
ERCB,42,89,131
GSE108109,6,105,111
GSE116626,7,74,81
GSE121233,5,5,10
GSE129973,20,20,40
GSE200828,6,147,153
GSE30122,50,19,69
GSE66494,8,53,61
GSE96804,20,41,61


In [11]:
def make_pipeline(model, genes, categorical):
    # PyCaret defaults: mean / mode imputation, one-hot encoding, no normalisation
    num = [('num', SimpleImputer(strategy='mean'), genes)] if genes else []
    cat = [('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)] if categorical else []
    return Pipeline([('pre', ColumnTransformer(num + cat)), ('model', clone(model))])

RS = RANDOM_STATE
# PyCaret 3.3.2 model library: ID -> (name, estimator with PyCaret's default args, in compare_models turbo)
PYCARET_MODELS = {
    'lr':       ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=RS), True),
    'knn':      ('K Neighbors Classifier', KNeighborsClassifier(n_jobs=-1), True),
    'nb':       ('Naive Bayes', GaussianNB(), True),
    'dt':       ('Decision Tree Classifier', DecisionTreeClassifier(random_state=RS), True),
    'svm':      ('SVM - Linear Kernel', SGDClassifier(tol=0.001, loss='hinge', penalty='l2', eta0=0.001,
                                                      random_state=RS, n_jobs=-1), True),
    'rbfsvm':   ('SVM - Radial Kernel', SVC(gamma='auto', C=1.0, probability=True, kernel='rbf',
                                            random_state=RS), False),
    'gpc':      ('Gaussian Process Classifier', GaussianProcessClassifier(copy_X_train=False, random_state=RS,
                                                                          n_jobs=-1), False),
    'mlp':      ('MLP Classifier', MLPClassifier(random_state=RS, max_iter=500), False),
    'ridge':    ('Ridge Classifier', RidgeClassifier(random_state=RS), True),
    'rf':       ('Random Forest Classifier', RandomForestClassifier(random_state=RS, n_jobs=-1), True),
    'qda':      ('Quadratic Discriminant Analysis', QuadraticDiscriminantAnalysis(), True),
    'ada':      ('Ada Boost Classifier', AdaBoostClassifier(random_state=RS), True),
    'gbc':      ('Gradient Boosting Classifier', GradientBoostingClassifier(random_state=RS), True),
    'lda':      ('Linear Discriminant Analysis', LinearDiscriminantAnalysis(), True),
    'et':       ('Extra Trees Classifier', ExtraTreesClassifier(random_state=RS, n_jobs=-1), True),
    'lightgbm': ('Light Gradient Boosting Machine', LGBMClassifier(random_state=RS, n_jobs=-1, verbose=-1), True),
    'dummy':    ('Dummy Classifier', DummyClassifier(strategy='prior', random_state=RS), True),
}

In [12]:
def positive_score(pipe, X_test):
    """Score for the Disease class: probability when available, else the decision function."""
    if hasattr(pipe, 'predict_proba'):
        return pipe.predict_proba(X_test)[:, 1]
    return pipe.decision_function(X_test)

def lodo(genes, categorical):
    """LODO: one fold per cohort; every sample gets exactly one out-of-fold prediction per model."""
    X = df[genes + categorical]
    logo = LeaveOneGroupOut()
    oof = []
    with warnings.catch_warnings():
        # expected on these data: QDA collinearity (one-hot columns), SAMME.R deprecation in AdaBoost
        warnings.filterwarnings('ignore', category=UserWarning)
        warnings.filterwarnings('ignore', category=FutureWarning)
        for i, (mid, (name, est, _)) in enumerate(PYCARET_MODELS.items(), 1):
            pipe = make_pipeline(est, genes, categorical)
            for train_idx, test_idx in logo.split(X, y, groups):
                held_out = np.unique(groups[test_idx])
                assert len(held_out) == 1 and held_out[0] not in set(groups[train_idx])
                pipe.fit(X.iloc[train_idx], y[train_idx])
                X_test = X.iloc[test_idx]
                oof.append(pd.DataFrame({'Model_ID': mid, 'Cohort': held_out[0],
                                         'GSM': df['GSM'].iloc[test_idx].to_numpy(), 'y_true': y[test_idx],
                                         'score': positive_score(pipe, X_test), 'y_pred': pipe.predict(X_test)}))
    oof = pd.concat(oof, ignore_index=True)
    assert (oof.groupby('Model_ID').size() == len(df)).all()
    assert not oof.duplicated(['Model_ID', 'GSM']).any()
    return oof

In [13]:
def metrics(d):
    return {'F1': f1_score(d['y_true'], d['y_pred'], zero_division=0),
            'Precision': precision_score(d['y_true'], d['y_pred'], zero_division=0),
            'Recall': recall_score(d['y_true'], d['y_pred'], zero_division=0),
            'Accuracy': accuracy_score(d['y_true'], d['y_pred']),
            'ROC-AUC': roc_auc_score(d['y_true'], d['score']) if d['y_true'].nunique() == 2 else np.nan}

METRICS = ['F1', 'Precision', 'Recall', 'Accuracy', 'ROC-AUC']

def lodo_metrics(oof):
    per_cohort = pd.DataFrame([{'Model_ID': mid, 'Cohort': c, 'n': len(d),
                                'n_control': int((d['y_true'] == 0).sum()), 'n_disease': int((d['y_true'] == 1).sum()),
                                **metrics(d)}
                               for (mid, c), d in oof.groupby(['Model_ID', 'Cohort'], sort=False)])

    # One value per metric per model: mean over the left-out cohorts (as PyCaret averages over folds)
    means = per_cohort.groupby('Model_ID', sort=False)[METRICS].mean()
    summary = pd.DataFrame({
        'Model': [PYCARET_MODELS[m][0] for m in means.index],
        'PyCaret ID': means.index,
        'In compare_models (turbo)': [PYCARET_MODELS[m][2] for m in means.index],
        **{m: means[m].to_numpy() for m in METRICS}})
    summary = summary.sort_values('ROC-AUC', ascending=False, ignore_index=True, kind='stable')
    return per_cohort, summary

In [14]:
import textwrap

def plot_best_roc(oof, summary, features, out_path, by='ROC-AUC'):
    # Mean ROC curve of the best model by mean `by` (F1 or ROC-AUC; the dummy baseline excluded): the per-cohort
    # curves averaged vertically at fixed false positive rates. Predictions are not pooled across folds - each
    # fold's model has its own score scale.
    ranked = summary[summary['PyCaret ID'] != 'dummy'].sort_values(by, ascending=False, kind='stable')
    best_id = ranked['PyCaret ID'].iloc[0]
    best_f1, best_auc = ranked[['F1', 'ROC-AUC']].iloc[0]
    fpr_grid = np.linspace(0, 1, 201)
    tprs, aucs, skipped = [], [], []
    for cohort, d in oof[oof['Model_ID'] == best_id].groupby('Cohort', sort=False):
        if d['y_true'].nunique() < 2:            # ROC undefined with one class
            skipped.append(cohort)
            continue
        fpr, tpr, _ = roc_curve(d['y_true'], d['score'])
        tpr_grid = np.interp(fpr_grid, fpr, tpr)
        tpr_grid[0] = 0.0
        tprs.append(tpr_grid)
        aucs.append(roc_auc_score(d['y_true'], d['score']))
    tpr_mean = np.mean(tprs, axis=0)
    tpr_mean[-1] = 1.0

    fig, ax = plt.subplots(figsize=(9, 10.5))   # larger canvas for the doubled font sizes
    ax.plot(fpr_grid, tpr_mean, lw=2.5, color='#0000FF',
            label=f'Mean of {len(aucs)} left-out cohorts: AUC {np.mean(aucs):.2f}')
    ax.plot([0, 1], [0, 1], ls='--', lw=1, color='grey', label='Chance')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect('equal')
    ax.set_xlabel('False positive rate (1 - specificity)', fontsize=20)
    ax.set_ylabel('True positive rate (sensitivity)', fontsize=20)
    ax.tick_params(labelsize=20)
    ax.set_title(f'{features}\nmean ROC of the best model by {by}:\n'
                 f'{PYCARET_MODELS[best_id][0]} ({best_id})\n'
                 f'mean F1 {best_f1:.3f}, mean ROC-AUC {best_auc:.3f}', fontsize=20)
    ax.legend(fontsize=16, loc='lower right')
    note = 'Average of the per-cohort ROC curves; each cohort scored by the model trained on the others.'
    if skipped:
        note += f' Not included (one class only): {", ".join(skipped)}.'
    fig.text(0.01, 0.005, textwrap.fill(note, 85), fontsize=14, color='grey')
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    out_path = out_path.with_name(f'{out_path.stem}_{best_auc:.3f}{out_path.suffix}')   # AUC from the title
    fig.savefig(out_path, dpi=200)
    plt.close(fig)   # 31 figures: written to disk, not displayed
    return best_id, out_path

In [15]:
def run_subset(genes):
    oof = lodo(genes, CATEGORICAL)
    _, summary = lodo_metrics(oof)
    features = f"{' + '.join(genes)} + {', '.join(CATEGORICAL)}"
    out_path = OUT_DIR / f"ROC_AUC_{len(genes)}genes_{'_'.join(genes)}.png"
    best_id, out_path = plot_best_roc(oof, summary, features, out_path)
    best = summary[summary['PyCaret ID'] == best_id].iloc[0]
    return {'Genes': ' + '.join(genes), 'n_genes': len(genes), 'Best model': best['Model'], 'PyCaret ID': best_id,
            **{m: round(best[m], 3) for m in ['Accuracy', 'F1', 'Precision', 'Recall', 'ROC-AUC']}, 'Image': out_path.name}

SUBSETS = [list(c) for k in range(1, len(GENES) + 1) for c in combinations(GENES, k)]
assert len(SUBSETS) == 31
for f in OUT_DIR.glob('ROC_AUC_*genes_*.png'):   # previous run's images: their names carry old AUCs
    f.unlink()
rows = []
for i, genes in enumerate(SUBSETS, 1):
    rows.append(run_subset(genes))
    print(f"[{i}/{len(SUBSETS)}] {rows[-1]['Genes']}: {rows[-1]['Best model']} ROC-AUC {rows[-1]['ROC-AUC']:.3f}", flush=True)

results = pd.DataFrame(rows).sort_values(['n_genes', 'ROC-AUC'], ascending=[True, False], ignore_index=True)
results.to_excel(OUT_DIR / 'ROC_AUC_summary.xlsx', index=False)
assert len(list(OUT_DIR.glob('ROC_AUC_*genes_*.png'))) == len(SUBSETS)
assert all((OUT_DIR / f).exists() for f in results['Image'])
results

[1/31] C4BPA: Naive Bayes ROC-AUC 0.576
[2/31] COL13A1: SVM - Linear Kernel ROC-AUC 0.561
[3/31] HABP2: Random Forest Classifier ROC-AUC 0.638
[4/31] CFH: Naive Bayes ROC-AUC 0.579
[5/31] CFI: Naive Bayes ROC-AUC 0.587
[6/31] C4BPA + COL13A1: Light Gradient Boosting Machine ROC-AUC 0.587
[7/31] C4BPA + HABP2: Light Gradient Boosting Machine ROC-AUC 0.609
[8/31] C4BPA + CFH: Naive Bayes ROC-AUC 0.585
[9/31] C4BPA + CFI: Gradient Boosting Classifier ROC-AUC 0.585
[10/31] COL13A1 + HABP2: SVM - Linear Kernel ROC-AUC 0.602
[11/31] COL13A1 + CFH: Logistic Regression ROC-AUC 0.577
[12/31] COL13A1 + CFI: Light Gradient Boosting Machine ROC-AUC 0.596
[13/31] HABP2 + CFH: K Neighbors Classifier ROC-AUC 0.604
[14/31] HABP2 + CFI: Random Forest Classifier ROC-AUC 0.666
[15/31] CFH + CFI: Light Gradient Boosting Machine ROC-AUC 0.624
[16/31] C4BPA + COL13A1 + HABP2: Random Forest Classifier ROC-AUC 0.621
[17/31] C4BPA + COL13A1 + CFH: Light Gradient Boosting Machine ROC-AUC 0.600
[18/31] C4BPA + C

,Genes,n_genes,Best model,PyCaret ID,Accuracy,F1,Precision,Recall,ROC-AUC,Image
0,HABP2,1,Random Forest Classifier,rf,0.668,0.749,0.737,0.803,0.638,ROC_AUC_1genes_HABP2.png
1,CFI,1,Naive Bayes,nb,0.732,0.790,0.726,0.904,0.587,ROC_AUC_1genes_CFI.png
2,CFH,1,Naive Bayes,nb,0.732,0.790,0.726,0.904,0.579,ROC_AUC_1genes_CFH.png
3,C4BPA,1,Naive Bayes,nb,0.732,0.790,0.726,0.904,0.576,ROC_AUC_1genes_C4BPA.png
4,COL13A1,1,SVM - Linear Kernel,svm,0.677,0.749,0.702,0.864,0.561,ROC_AUC_1genes_COL13A1.png
5,HABP2 + CFI,2,Random Forest Classifier,rf,0.701,0.785,0.732,0.895,0.666,ROC_AUC_2genes_HABP2_CFI.png
6,CFH + CFI,2,Light Gradient Boosting Machine,lightgbm,0.685,0.781,0.736,0.892,0.624,ROC_AUC_2genes_CFH_CFI.png
7,C4BPA + HABP2,2,Light Gradient Boosting Machine,lightgbm,0.694,0.782,0.733,0.897,0.609,ROC_AUC_2genes_C4BPA_HABP2.png
8,HABP2 + CFH,2,K Neighbors Classifier,knn,0.704,0.793,0.731,0.921,0.604,ROC_AUC_2genes_HABP2_CFH.png
9,COL13A1 + HABP2,2,SVM - Linear Kernel,svm,0.640,0.714,0.731,0.798,0.602,ROC_AUC_2genes_COL13A1_HABP2.png


In [16]:
with open(OUT_DIR / 'ROC_AUC_sessionInfo.txt', 'w') as fh:
    fh.write(f'python {sys.version}\nplatform {platform.platform()}\n')
    fh.write(f'scikit-learn {sklearn.__version__}\nlightgbm {lightgbm.__version__}\n'
             f'pandas {pd.__version__}\nnumpy {np.__version__}\n')
    fh.write(f'RANDOM_STATE={RANDOM_STATE} MERGE_NEPTUNE={MERGE_NEPTUNE}\n')
    fh.write(f'models (PyCaret 3.3.2 defaults): {", ".join(PYCARET_MODELS)}\n')
    fh.write(f'input {DATA_PATH.resolve()} ({len(df)} samples, {df["Cohort"].nunique()} LODO groups)\n')
    fh.write(f'features: every non-empty subset of {GENES} ({len(SUBSETS)} subsets) + {CATEGORICAL}; '
             'best model per subset by mean LODO ROC-AUC (dummy excluded)\n')